# load dependecies

In [ ]:
## Initialzing and loading required libraries and subfunctions
import numpy as np
import matplotlib.pyplot as plt
import copy
import yasa
from mne.filter import resample
import pynapple as nap
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import normalize
import requests
from io import BytesIO
import sails
from ssqueezepy import cwt
import re
from scipy.stats import entropy
import os, sys

import scipy
from scipy import signal
from scipy.interpolate import griddata
from scipy.signal import correlate
from scipy.stats import pearsonr
from scipy.fft import fft
from scipy.spatial.distance import euclidean
from scipy.signal import spectrogram
from scipy.io import loadmat
import scipy.fft
import scipy.stats
import scipy.io as sio
from scipy.signal import hilbert

import emd as emd
import emd.sift as sift
import emd.spectra as spectra

from neurodsp.sim import sim_combined
from neurodsp.plts import plot_time_series, plot_timefrequency
from neurodsp.utils import create_times
from neurodsp.timefrequency.wavelets import compute_wavelet_transform
from neurodsp.filt import filter_signal

# Load required libraries
import numpy as np
from scipy.io import loadmat
from scipy.signal import hilbert
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns
from neurodsp.filt import filter_signal, filter_signal_fir, design_fir_filter
import emd
import pandas as pd
from sklearn.preprocessing import Normalizer
from tqdm import tqdm
import plotly.express as px
import copy
import umap.umap_ as umap
from scipy.spatial import cKDTree
import pickle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
try:
    from mne_connectivity import spectral_connectivity_epochs
    _HAS_MNE_GC = True
except Exception:
    _HAS_MNE_GC = False
    print('mne-connectivity is not available in this Python environment.')
    print('Install with: pip install mne-connectivity mne')


## UTILS

for rel in ('src', '../src'):
    p = os.path.abspath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from utils import *
from detect_pt import *

from scipy.io import loadmat
import numpy as np
from neurodsp.filt import filter_signal
import copy
import emd
from scipy.spatial import cKDTree
from tqdm import tqdm

sns.set(style='white', context='notebook')

In [ ]:
path_to_config = '/Users/amir/Desktop/for Abdel/emd_masksift_CA1_config.yml'
config = emd.sift.SiftConfig.from_yaml_file(path_to_config)

# preprocessing

In [ ]:
def extract_cycle_info(imfs, imf_frequencies):

  waveforms = pd.DataFrame()
  all_trials = pd.DataFrame()
  raw_wavelets = []
  all_FPPs = []

  theta_range = [5, 12]
  frequencies = np.arange(15, 141, 1)  # Logarithmically spaced frequencies from 15 to 300 Hz
  angles=np.linspace(-180,180,19)
  fs = 1000

  for idx, imf in enumerate(imfs):
    cycle_data = get_cycle_data(imf[:, 5], fs)

    amp_thresh = np.percentile(cycle_data['IA'], 25) # higher than 25th percentile of the data
    lo_freq_duration = fs/5  # restrict the analysis to 5-12 Hz
    hi_freq_duration = fs/12

    conditions = ['is_good==1',
                        f'duration_samples<{lo_freq_duration}',
                        f'duration_samples>{hi_freq_duration}',
                        f'max_amp>{amp_thresh}']
    print(len(cycle_data['theta_imf']))
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    
    # Check if any cycles satisfy the conditions
    if all_cycles is None or all_cycles.chain_vect.size == 0:
        print("No cycles satisfy the conditions.")
        return pd.DataFrame(), pd.DataFrame(), []
    
    subset_cycles_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_cycles_df['index'].values

    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycles_inds = arrange_cycle_inds(all_cycles_inds)

    freqs = imf_frequencies[idx]
    sub_theta, theta, supra_theta = tg_split(freqs, theta_range)
    supra_theta_sig = np.sum(imf.T[supra_theta], axis=0)

    # # Corrected Wavelet Transform Computation
    raw_data=sails.wavelet.morlet(supra_theta_sig, freqs=frequencies, sample_rate=fs, ncycles=5,ret_mode='power', normalise='simple')
    raw_wavelets.append(raw_data)
    supraPlot = scipy.stats.zscore(raw_data, axis=1)
    FPP = bin_tf_to_fpp(cycles_inds, supraPlot, bin_count=19)
    all_FPPs.append(FPP)

    # Compute mode frequency for each cycle
    mode_freqs, entropies = compute_mode_frequency_and_entropy(FPP, frequencies, angles)

    all_waveforms, _ = emd.cycles.phase_align(cycle_data['IP'], cycle_data['theta_imf'],
                                                            cycles=all_cycles.iterate(through='subset'), npoints=100)
    all_waveforms = pd.DataFrame(all_waveforms.T)

    waveforms = pd.concat([waveforms, all_waveforms])

    trial = all_cycles.get_metric_dataframe(subset=True)
    trial['mode_freqs'] = mode_freqs
    trial['entropy'] = entropies
    all_trials = pd.concat([all_trials, trial])

  return waveforms, all_trials, all_FPPs

## TODO

extract one hpc and pfc posttrial 5.
then try to extract p/t from hpc

for all p/t extract corrsoponding pfc parts

do a granger casuality between them

In [ ]:
path_to_hpc = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/HPC_100_CH4_0.continuous.mat'
path_to_pfc = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/PFC_100_CH47_0.continuous.mat'
path_to_hypno = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/2018-08-01_15-49-15_Post-Trial5_concatenated-states.mat'

In [ ]:
fs = 1000
theta_range = [5, 12]

In [ ]:
lfpHPC, hypno, _ = get_data(path_to_hpc, path_to_hypno)

In [ ]:
lfpPFC, _, _ = get_data(path_to_pfc, path_to_hypno, type='pfc')

In [ ]:
phasic_interval, tonic_interval, lfp_raw = extract_pt_intervals(lfpHPC, hypno, fs=1000)

In [ ]:
tonic_imfs, tonic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, tonic_interval, config, return_imfs_freqs=True)
phasic_imfs, phasic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, phasic_interval, config, return_imfs_freqs=True)

In [ ]:
tonic_theta_imfs  = [imf[:, 5] for imf in tonic_imfs]
phasic_theta_imfs = [imf[:, 5] for imf in phasic_imfs]

In [ ]:
phasic_waveforms, phasic_trials, phasic_fpps = extract_cycle_info(phasic_imfs, phasic_freqs)
tonic_waveforms, tonic_trials, tonic_fpps = extract_cycle_info(tonic_imfs, tonic_freqs)

In [ ]:
phasic_waveforms

In [ ]:
def extract_lfp_by_pt_intervals(lfp, fs, interval):
    rem_lfp = []
    for ii in range(len(interval)):
        start_idx = int(interval.loc[ii, 'start'] * fs)
        end_idx = int(interval.loc[ii, 'end'] * fs)
        rem_lfp.append(np.array(lfp[start_idx:end_idx]))
    return rem_lfp


In [ ]:
pfc_phasic_rem_lfp = extract_lfp_by_pt_intervals(lfpPFC, fs, phasic_interval)
pfc_tonic_rem_lfp = extract_lfp_by_pt_intervals(lfpPFC, fs, tonic_interval)

In [ ]:
theta_pfc_phasic_rem_lfp = []
for sig in pfc_phasic_rem_lfp:
    filtered = bandpass_filter(sig, fs=1000, low=5, high=12, order=4)
    theta_pfc_phasic_rem_lfp.append(filtered)

theta_pfc_tonic_rem_lfp = []
for sig in pfc_tonic_rem_lfp:
    filtered = bandpass_filter(sig, fs=1000, low=5, high=12, order=4)
    theta_pfc_tonic_rem_lfp.append(filtered)

## Granger Causality (Theta, HPC vs PFC)

This section computes bidirectional Granger causality per matched phasic/tonic segment:
- `HPC -> PFC`
- `PFC -> HPC`

Outputs:
- `gc_results`: per-segment best-lag results
- `gc_summary`: state/direction summary with nominal + FDR significance
- `gc_lag_pvalues`: lag-level p-values

In [ ]:
from scipy.stats import f as f_dist


def _bh_fdr_flags(pvals, alpha=0.05):
    """Benjamini-Hochberg FDR flags for a 1D array of p-values."""
    p = np.asarray(pvals, dtype=float)
    valid = np.isfinite(p)
    flags = np.zeros_like(p, dtype=bool)
    if valid.sum() == 0:
        return flags

    pv = p[valid]
    m = len(pv)
    order = np.argsort(pv)
    ranked = pv[order]
    thresh = alpha * (np.arange(1, m + 1) / m)
    passed = ranked <= thresh

    if np.any(passed):
        k = np.where(passed)[0].max()
        cutoff = ranked[k]
        flags[valid] = pv <= cutoff

    return flags


def _lag_matrix(x, lag):
    """Columns: x(t-1), ..., x(t-lag), aligned to x[lag:]."""
    x = np.asarray(x, dtype=float)
    return np.column_stack([x[lag - k:-k] for k in range(1, lag + 1)])


def _ols_ssr(y, X):
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    return float(resid.T @ resid)


def _granger_ssr_ftest(target, driver, lag):
    """
    Null: driver does NOT Granger-cause target (at this lag).
    SSR-based F-test.
    """
    target = np.asarray(target, dtype=float)
    driver = np.asarray(driver, dtype=float)

    if len(target) <= (2 * lag + 5):
        return np.nan, np.nan, np.nan, np.nan

    y = target[lag:]
    y_lags = _lag_matrix(target, lag)
    x_lags = _lag_matrix(driver, lag)

    Xr = np.column_stack([np.ones(len(y)), y_lags])
    Xu = np.column_stack([np.ones(len(y)), y_lags, x_lags])

    ssr_r = _ols_ssr(y, Xr)
    ssr_u = _ols_ssr(y, Xu)

    df_num = lag
    df_den = len(y) - Xu.shape[1]

    if df_den <= 0 or not np.isfinite(ssr_r) or not np.isfinite(ssr_u):
        return np.nan, np.nan, float(df_num), float(df_den)

    num = max(ssr_r - ssr_u, 0.0) / df_num
    den = ssr_u / df_den
    if den <= 0:
        return np.nan, np.nan, float(df_num), float(df_den)

    f_stat = num / den
    p_val = float(f_dist.sf(f_stat, df_num, df_den))
    return float(f_stat), p_val, float(df_num), float(df_den)


def _prepare_pair(hpc_sig, pfc_sig, difference=True):
    """Match length, drop non-finite, optional first-difference."""
    h = np.asarray(hpc_sig, dtype=float).squeeze()
    p = np.asarray(pfc_sig, dtype=float).squeeze()

    n = min(len(h), len(p))
    if n < 30:
        return None, None

    h = h[:n]
    p = p[:n]

    valid = np.isfinite(h) & np.isfinite(p)
    h = h[valid]
    p = p[valid]

    if difference:
        h = np.diff(h)
        p = np.diff(p)

    if min(len(h), len(p)) < 30:
        return None, None

    return h, p


def _segment_gc(hpc_theta, pfc_theta, maxlag=20, difference=True):
    """Bidirectional GC for one segment."""
    h, p = _prepare_pair(hpc_theta, pfc_theta, difference=difference)
    if h is None:
        return None

    lag_use = min(int(maxlag), max(1, len(h) // 10))
    if lag_use < 1:
        return None

    def _run(direction):
        if direction == 'HPC->PFC':
            target, driver = p, h
        elif direction == 'PFC->HPC':
            target, driver = h, p
        else:
            raise ValueError('Unknown direction')

        rows = []
        for lag in range(1, lag_use + 1):
            f_stat, p_val, df_num, df_den = _granger_ssr_ftest(target, driver, lag)
            rows.append((lag, f_stat, p_val, df_num, df_den))

        lag_df = pd.DataFrame(rows, columns=['lag', 'f_stat', 'p_value', 'df_num', 'df_denom'])
        lag_df = lag_df[np.isfinite(lag_df['p_value'])].copy()
        if lag_df.empty:
            return None

        best = lag_df.loc[lag_df['p_value'].idxmin()]
        return {
            'direction': direction,
            'tested_maxlag': int(lag_use),
            'best_lag': int(best['lag']),
            'best_f_stat': float(best['f_stat']),
            'min_p_value': float(best['p_value']),
            'n_samples_used': int(len(h)),
            'lag_table': lag_df,
        }

    out = []
    for d in ('HPC->PFC', 'PFC->HPC'):
        res = _run(d)
        if res is not None:
            out.append(res)

    return out if out else None


def run_granger_by_state(hpc_theta_list, pfc_theta_list, state_name, maxlag=20, difference=True):
    """Run bidirectional GC across all matched segments in one state."""
    n_h = len(hpc_theta_list)
    n_p = len(pfc_theta_list)
    n = min(n_h, n_p)

    if n_h != n_p:
        print(f"[{state_name}] warning: segment count mismatch (HPC={n_h}, PFC={n_p}). Using first {n} matched segments.")

    rows = []
    lag_tables = []

    for seg_idx in range(n):
        seg_out = _segment_gc(hpc_theta_list[seg_idx], pfc_theta_list[seg_idx], maxlag=maxlag, difference=difference)
        if seg_out is None:
            continue

        for r in seg_out:
            rows.append({
                'state': state_name,
                'segment_idx': seg_idx,
                'direction': r['direction'],
                'tested_maxlag': r['tested_maxlag'],
                'best_lag': r['best_lag'],
                'best_f_stat': r['best_f_stat'],
                'min_p_value': r['min_p_value'],
                'n_samples_used': r['n_samples_used'],
            })

            lt = r['lag_table'].copy()
            lt['state'] = state_name
            lt['segment_idx'] = seg_idx
            lt['direction'] = r['direction']
            lag_tables.append(lt)

    if len(rows) == 0:
        empty = pd.DataFrame(columns=['state', 'segment_idx', 'direction', 'tested_maxlag', 'best_lag', 'best_f_stat', 'min_p_value', 'n_samples_used'])
        return empty, empty.copy(), pd.DataFrame()

    result_df = pd.DataFrame(rows)

    result_df['fdr_sig_0p05'] = False
    for d in result_df['direction'].unique():
        m = result_df['direction'] == d
        result_df.loc[m, 'fdr_sig_0p05'] = _bh_fdr_flags(result_df.loc[m, 'min_p_value'].values, alpha=0.05)

    summary_df = (
        result_df.groupby(['state', 'direction'], as_index=False)
        .agg(
            n_segments_tested=('segment_idx', 'count'),
            n_nominal_p_lt_0p05=('min_p_value', lambda x: int((np.asarray(x) < 0.05).sum())),
            n_fdr_sig_0p05=('fdr_sig_0p05', lambda x: int(np.asarray(x, dtype=bool).sum())),
            median_best_lag=('best_lag', 'median'),
            median_min_p=('min_p_value', 'median'),
            median_best_f=('best_f_stat', 'median'),
            median_n_samples=('n_samples_used', 'median'),
        )
    )

    lag_df = pd.concat(lag_tables, ignore_index=True) if lag_tables else pd.DataFrame()
    return result_df, summary_df, lag_df


In [ ]:
# Segment-count sanity check before running GC
print('Segment counts before GC:')
print(f"  phasic: HPC={len(phasic_theta_imfs)}  PFC={len(theta_pfc_phasic_rem_lfp)}")
print(f"  tonic : HPC={len(tonic_theta_imfs)}  PFC={len(theta_pfc_tonic_rem_lfp)}")

phasic_gc, phasic_gc_summary, phasic_lag_pvals = run_granger_by_state(
    phasic_theta_imfs,
    theta_pfc_phasic_rem_lfp,
    state_name='phasic',
    maxlag=20,
    difference=True,
)

tonic_gc, tonic_gc_summary, tonic_lag_pvals = run_granger_by_state(
    tonic_theta_imfs,
    theta_pfc_tonic_rem_lfp,
    state_name='tonic',
    maxlag=20,
    difference=True,
)

# Combined outputs
gc_results = pd.concat([phasic_gc, tonic_gc], ignore_index=True)
gc_summary = pd.concat([phasic_gc_summary, tonic_gc_summary], ignore_index=True)
gc_lag_pvalues = pd.concat([phasic_lag_pvals, tonic_lag_pvals], ignore_index=True)

print()
print('=== Granger Summary (theta signals) ===')
display(gc_summary.sort_values(['state', 'direction']).reset_index(drop=True))

print()
print('Per-segment best results (head):')
display(gc_results.head(20))


## Visualize GC Results

Plots: summary counts, per-segment significance strength, best-lag distribution, and lag-wise significance heatmaps.

In [ ]:
# Prepare plotting frame
plot_df = gc_results.copy()
plot_df['neglog10_min_p'] = -np.log10(np.clip(plot_df['min_p_value'].astype(float), 1e-300, 1.0))
plot_df['direction_label'] = plot_df['direction'].replace({'HPC->PFC': 'HPC -> PFC', 'PFC->HPC': 'PFC -> HPC'})

sum_df = gc_summary.copy()
sum_df['direction_label'] = sum_df['direction'].replace({'HPC->PFC': 'HPC -> PFC', 'PFC->HPC': 'PFC -> HPC'})

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

sns.barplot(data=sum_df, x='state', y='n_segments_tested', hue='direction_label', ax=axes[0, 0])
axes[0, 0].set_title('Segments Tested')
axes[0, 0].set_xlabel('state')
axes[0, 0].set_ylabel('count')

sns.barplot(data=sum_df, x='state', y='n_fdr_sig_0p05', hue='direction_label', ax=axes[0, 1])
axes[0, 1].set_title('FDR-significant Segments (q < 0.05)')
axes[0, 1].set_xlabel('state')
axes[0, 1].set_ylabel('count')

sns.boxplot(data=plot_df, x='state', y='neglog10_min_p', hue='direction_label', ax=axes[1, 0])
axes[1, 0].set_title('Per-segment GC Strength (-log10 min p)')
axes[1, 0].set_xlabel('state')
axes[1, 0].set_ylabel('-log10(min p)')

sns.stripplot(data=plot_df, x='state', y='best_lag', hue='direction_label', dodge=True, alpha=0.75, ax=axes[1, 1])
axes[1, 1].set_title('Best Lag per Segment')
axes[1, 1].set_xlabel('state')
axes[1, 1].set_ylabel('lag (samples)')

for ax in axes.flat:
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[:2], labels[:2], frameon=False, title='direction')

plt.show()

lag_df = gc_lag_pvalues.copy()
lag_df['sig_0p05'] = lag_df['p_value'] < 0.05
lag_frac = (
    lag_df.groupby(['state', 'direction', 'lag'], as_index=False)['sig_0p05']
    .mean()
    .rename(columns={'sig_0p05': 'frac_sig'})
)

h2p = lag_frac[lag_frac['direction'] == 'HPC->PFC'].pivot(index='state', columns='lag', values='frac_sig')
p2h = lag_frac[lag_frac['direction'] == 'PFC->HPC'].pivot(index='state', columns='lag', values='frac_sig')

fig, ax = plt.subplots(1, 2, figsize=(16, 4), constrained_layout=True)
sns.heatmap(h2p, cmap='viridis', vmin=0, vmax=1, cbar=True, ax=ax[0])
ax[0].set_title('Fraction Significant by Lag (HPC -> PFC)')
ax[0].set_xlabel('lag')
ax[0].set_ylabel('state')

sns.heatmap(p2h, cmap='viridis', vmin=0, vmax=1, cbar=True, ax=ax[1])
ax[1].set_title('Fraction Significant by Lag (PFC -> HPC)')
ax[1].set_xlabel('lag')
ax[1].set_ylabel('state')

plt.show()


## Version A: mne-connectivity spectral GC pipeline

This version now uses **raw interval-matched LFP** as inputs:
- HPC raw: extracted from `lfp_raw` with `phasic_interval/tonic_interval`
- PFC raw: `pfc_phasic_rem_lfp` / `pfc_tonic_rem_lfp`

GC is then computed spectrally, and theta (5-12 Hz) is used only for output summarization.

In [ ]:
# MNE spectral GC version (theta-focused, stability-tuned)
# Install if needed: pip install mne-connectivity mne

import warnings
from scipy.stats import zscore

try:
    from mne_connectivity import spectral_connectivity_epochs
    _HAS_MNE_GC = True
except Exception:
    _HAS_MNE_GC = False
    print('mne-connectivity is not available in this Python environment.')
    print('Install with: pip install mne-connectivity mne')


def _prepare_equal_length_epochs(
    hpc_list,
    pfc_list,
    sfreq,
    min_samples,
    min_std=1e-8,
):
    """Build equal-length epoch array: (n_epochs, 2, n_times)."""
    n = min(len(hpc_list), len(pfc_list))
    if n == 0:
        return None, None

    pairs = []
    for i in range(n):
        h = np.asarray(hpc_list[i], dtype=float).squeeze()
        p = np.asarray(pfc_list[i], dtype=float).squeeze()

        m = min(len(h), len(p))
        if m < min_samples:
            continue

        h = h[:m]
        p = p[:m]

        valid = np.isfinite(h) & np.isfinite(p)
        h = h[valid]
        p = p[valid]

        if min(len(h), len(p)) < min_samples:
            continue

        if (np.nanstd(h) < min_std) or (np.nanstd(p) < min_std):
            continue

        pairs.append((h, p))

    if len(pairs) == 0:
        return None, None

    # Equal-length requirement for spectral_connectivity_epochs
    common_len = min(min(len(h), len(p)) for h, p in pairs)
    if common_len < min_samples:
        return None, None

    epochs = np.zeros((len(pairs), 2, common_len), dtype=float)
    meta_rows = []

    for j, (h, p) in enumerate(pairs):
        h = h[:common_len]
        p = p[:common_len]

        # Within-epoch normalization for numerical stability
        h = zscore(h, ddof=1)
        p = zscore(p, ddof=1)

        if not (np.isfinite(h).all() and np.isfinite(p).all()):
            continue

        epochs[j, 0, :] = h
        epochs[j, 1, :] = p
        meta_rows.append({'pair_idx': j, 'n_samples': common_len, 'epoch_sec': common_len / sfreq})

    if len(meta_rows) == 0:
        return None, None

    # In rare cases some rows may have been skipped post-zscore checks
    epochs = epochs[:len(meta_rows)]
    return epochs, pd.DataFrame(meta_rows)


def run_mne_spectral_gc_for_state(
    hpc_list,
    pfc_list,
    state_name,
    sfreq,
    fmin=5.0,
    fmax=20.0,
    theta_band=(5.0, 12.0),
    gc_n_lags=10,
    min_epoch_sec=2.0,
    min_n_cycles=5,
):
    """Run spectral GC in both directions for one state."""
    if not _HAS_MNE_GC:
        return None, None, None

    # Reliability constraints
    min_samples_cycles = int(np.ceil((min_n_cycles * sfreq) / max(fmin, 1e-9)))
    min_samples_epoch = int(np.ceil(min_epoch_sec * sfreq))
    min_samples_lags = int(max(50, gc_n_lags * 15))
    min_samples = max(min_samples_cycles, min_samples_epoch, min_samples_lags)

    epochs, meta_df = _prepare_equal_length_epochs(
        hpc_list, pfc_list, sfreq=sfreq, min_samples=min_samples
    )

    if epochs is None:
        print(f'[{state_name}] no valid matched epochs for MNE spectral GC after quality filters.')
        return None, None, None

    n_times = epochs.shape[2]
    epoch_sec = n_times / sfreq

    # Ensure at least min_n_cycles at the lowest analyzed frequency
    fmin_eff = max(float(fmin), float(min_n_cycles) / max(epoch_sec, 1e-9))
    fmax_eff = float(fmax)
    if fmin_eff >= fmax_eff:
        print(f'[{state_name}] invalid freq window after constraints: fmin_eff={fmin_eff:.3f}, fmax={fmax_eff:.3f}.')
        return None, None, None

    # Keep lag order conservative vs epoch length
    gc_lags_eff = int(min(gc_n_lags, max(3, n_times // 50)))

    idx_h2p = (np.array([[0]]), np.array([[1]]))
    idx_p2h = (np.array([[1]]), np.array([[0]]))

    # Suppress low-level determinant warnings; we'll validate outputs explicitly
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='.*divide by zero encountered in det.*', category=RuntimeWarning)
        warnings.filterwarnings('ignore', message='.*invalid value encountered in det.*', category=RuntimeWarning)

        con_h2p = spectral_connectivity_epochs(
            epochs,
            method='gc',
            indices=idx_h2p,
            sfreq=sfreq,
            mode='multitaper',
            fmin=fmin_eff,
            fmax=fmax_eff,
            faverage=False,
            gc_n_lags=gc_lags_eff,
            n_jobs=1,
            verbose=False,
        )

        con_p2h = spectral_connectivity_epochs(
            epochs,
            method='gc',
            indices=idx_p2h,
            sfreq=sfreq,
            mode='multitaper',
            fmin=fmin_eff,
            fmax=fmax_eff,
            faverage=False,
            gc_n_lags=gc_lags_eff,
            n_jobs=1,
            verbose=False,
        )

    freqs = np.asarray(con_h2p.freqs, dtype=float)
    h2p = np.squeeze(con_h2p.get_data()).astype(float)
    p2h = np.squeeze(con_p2h.get_data()).astype(float)

    valid_spec = np.isfinite(h2p) & np.isfinite(p2h)
    if valid_spec.sum() == 0:
        print(f'[{state_name}] spectral GC returned no finite points.')
        return None, None, None

    theta_lo, theta_hi = theta_band
    theta_mask = (freqs >= theta_lo) & (freqs <= theta_hi) & valid_spec

    theta_h2p = float(np.nanmean(h2p[theta_mask])) if np.any(theta_mask) else np.nan
    theta_p2h = float(np.nanmean(p2h[theta_mask])) if np.any(theta_mask) else np.nan

    summary_row = pd.DataFrame([{
        'state': state_name,
        'n_epochs_used': int(epochs.shape[0]),
        'n_samples_per_epoch': int(n_times),
        'epoch_sec': float(epoch_sec),
        'fmin_eff_hz': float(fmin_eff),
        'fmax_eff_hz': float(fmax_eff),
        'gc_n_lags_eff': int(gc_lags_eff),
        'theta_gc_hpc_to_pfc': theta_h2p,
        'theta_gc_pfc_to_hpc': theta_p2h,
        'theta_directionality_h2p_minus_p2h': theta_h2p - theta_p2h,
        'n_valid_freq_bins': int(valid_spec.sum()),
    }])

    spec_dict = {
        'state': state_name,
        'freqs': freqs,
        'gc_hpc_to_pfc': h2p,
        'gc_pfc_to_hpc': p2h,
        'valid_spec_mask': valid_spec,
        'meta': meta_df,
    }

    return summary_row, spec_dict, epochs


if _HAS_MNE_GC:
    # Raw interval segments (not theta-filtered inputs)
    hpc_phasic_raw_lfp = extract_lfp_by_pt_intervals(lfp_raw, fs, phasic_interval)
    hpc_tonic_raw_lfp = extract_lfp_by_pt_intervals(lfp_raw, fs, tonic_interval)

    # PFC raw segments were already extracted earlier in the notebook
    pfc_phasic_raw_lfp = pfc_phasic_rem_lfp
    pfc_tonic_raw_lfp = pfc_tonic_rem_lfp

    print('MNE input segment counts (raw):')
    print(f"  phasic: HPC={len(hpc_phasic_raw_lfp)}  PFC={len(pfc_phasic_raw_lfp)}")
    print(f"  tonic : HPC={len(hpc_tonic_raw_lfp)}  PFC={len(pfc_tonic_raw_lfp)}")

    mne_rows = []
    mne_specs = {}

    out_ph = run_mne_spectral_gc_for_state(
        hpc_phasic_raw_lfp,
        pfc_phasic_raw_lfp,
        'phasic',
        sfreq=fs,
        fmin=5.0,
        fmax=20.0,
        theta_band=(5.0, 12.0),
        gc_n_lags=10,
        min_epoch_sec=2.0,
        min_n_cycles=5,
    )

    out_to = run_mne_spectral_gc_for_state(
        hpc_tonic_raw_lfp,
        pfc_tonic_raw_lfp,
        'tonic',
        sfreq=fs,
        fmin=5.0,
        fmax=20.0,
        theta_band=(5.0, 12.0),
        gc_n_lags=10,
        min_epoch_sec=2.0,
        min_n_cycles=5,
    )

    for out in [out_ph, out_to]:
        row, spec, _epochs = out
        if row is not None:
            mne_rows.append(row)
            mne_specs[spec['state']] = spec

    mne_gc_summary = pd.concat(mne_rows, ignore_index=True) if mne_rows else pd.DataFrame()

    print('MNE spectral GC summary (theta-band averages from raw-input GC):')
    display(mne_gc_summary)

    if not mne_gc_summary.empty:
        fig, ax = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)

        for state_name, sd in mne_specs.items():
            vm = sd['valid_spec_mask']
            ax[0].plot(sd['freqs'][vm], sd['gc_hpc_to_pfc'][vm], label=f'{state_name}: HPC->PFC', linewidth=2)
            ax[0].plot(sd['freqs'][vm], sd['gc_pfc_to_hpc'][vm], label=f'{state_name}: PFC->HPC', linewidth=2, linestyle='--')
        ax[0].axvspan(5, 12, color='gray', alpha=0.15, label='theta band')
        ax[0].set_title('MNE Spectral GC (raw inputs, valid bins only)')
        ax[0].set_xlabel('Frequency (Hz)')
        ax[0].set_ylabel('GC')
        ax[0].legend(frameon=False)

        bar_df = mne_gc_summary.melt(
            id_vars=['state'],
            value_vars=['theta_gc_hpc_to_pfc', 'theta_gc_pfc_to_hpc'],
            var_name='direction',
            value_name='theta_gc'
        )
        bar_df['direction'] = bar_df['direction'].replace({
            'theta_gc_hpc_to_pfc': 'HPC -> PFC',
            'theta_gc_pfc_to_hpc': 'PFC -> HPC'
        })
        sns.barplot(data=bar_df, x='state', y='theta_gc', hue='direction', ax=ax[1])
        ax[1].set_title('Theta-Band GC (5-12 Hz, from raw-input model)')
        ax[1].set_xlabel('state')
        ax[1].set_ylabel('mean GC')
        ax[1].legend(frameon=False)

        plt.show()


## Version B: statsmodels VAR GC pipeline

Uses `statsmodels.tsa.api.VAR` and `test_causality` per matched segment on your theta signals
(`HPC theta from IMF`, `PFC theta from bandpass`).

This section auto-selects lag by BIC (bounded by `maxlags`) and reports bidirectional p-values.

In [ ]:
# statsmodels VAR version
# Install if needed: pip install statsmodels

import warnings

try:
    from statsmodels.tsa.api import VAR
    try:
        from statsmodels.tools.sm_exceptions import ValueWarning as SMValueWarning
    except Exception:
        SMValueWarning = ValueWarning
    _HAS_STATSMODELS = True
except Exception as e:
    _HAS_STATSMODELS = False
    print('statsmodels is not available in this Python environment.')
    print('Install with: pip install statsmodels')


def _bh_fdr_flags_local(pvals, alpha=0.05):
    p = np.asarray(pvals, dtype=float)
    valid = np.isfinite(p)
    out = np.zeros_like(p, dtype=bool)
    if valid.sum() == 0:
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    thresh = alpha * (np.arange(1, m + 1) / m)
    passed = ranked <= thresh
    if np.any(passed):
        k = np.where(passed)[0].max()
        cutoff = ranked[k]
        out[valid] = pv <= cutoff
    return out


def _prepare_var_pair(hpc_sig, pfc_sig, difference=True, min_samples=80):
    h = np.asarray(hpc_sig, dtype=float).squeeze()
    p = np.asarray(pfc_sig, dtype=float).squeeze()
    n = min(len(h), len(p))
    if n < min_samples:
        return None

    h = h[:n]
    p = p[:n]
    valid = np.isfinite(h) & np.isfinite(p)
    h = h[valid]
    p = p[valid]

    df = pd.DataFrame({'hpc': h, 'pfc': p})
    if difference:
        df = df.diff().dropna()

    if len(df) < min_samples:
        return None

    # Standardize columns for numerical stability
    df = (df - df.mean()) / (df.std(ddof=1) + 1e-12)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    if len(df) < min_samples:
        return None

    # Force simple supported index (avoids unsupported-index warnings)
    df = df.reset_index(drop=True)

    return df


def run_statsmodels_var_gc_by_state(hpc_list, pfc_list, state_name, maxlags=20, ic='bic', difference=True):
    if not _HAS_STATSMODELS:
        return pd.DataFrame(), pd.DataFrame()

    n = min(len(hpc_list), len(pfc_list))
    if len(hpc_list) != len(pfc_list):
        print(f'[{state_name}] warning: segment count mismatch (HPC={len(hpc_list)}, PFC={len(pfc_list)}). Using first {n} matched segments.')

    rows = []

    for i in range(n):
        df = _prepare_var_pair(hpc_list[i], pfc_list[i], difference=difference, min_samples=80)
        if df is None:
            continue

        maxlags_use = min(int(maxlags), max(1, len(df) // 5))

        try:
            # Silence only the unsupported-index forecast warning (not relevant for GC testing)
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    'ignore',
                    message='An unsupported index was provided.*',
                    category=SMValueWarning
                )

                model = VAR(df)
                sel = model.select_order(maxlags=maxlags_use)
                chosen = getattr(sel, ic)
                if chosen is None or (isinstance(chosen, float) and np.isnan(chosen)):
                    lag = 1
                else:
                    lag = int(chosen)
                    lag = max(1, min(lag, maxlags_use))

                fit = model.fit(lag)

                # HPC -> PFC means: does hpc help predict pfc?
                test_h2p = fit.test_causality('pfc', ['hpc'], kind='f')
                # PFC -> HPC
                test_p2h = fit.test_causality('hpc', ['pfc'], kind='f')

            rows.append({
                'state': state_name,
                'segment_idx': i,
                'n_samples_used': int(len(df)),
                'selected_lag': int(lag),
                'f_hpc_to_pfc': float(np.squeeze(test_h2p.test_statistic)),
                'p_hpc_to_pfc': float(np.squeeze(test_h2p.pvalue)),
                'f_pfc_to_hpc': float(np.squeeze(test_p2h.test_statistic)),
                'p_pfc_to_hpc': float(np.squeeze(test_p2h.pvalue)),
            })
        except Exception:
            continue

    res = pd.DataFrame(rows)
    if res.empty:
        return res, pd.DataFrame()

    # Long format for summary/FDR
    long_df = pd.concat([
        res[['state', 'segment_idx', 'n_samples_used', 'selected_lag', 'f_hpc_to_pfc', 'p_hpc_to_pfc']]
           .rename(columns={'f_hpc_to_pfc': 'f_stat', 'p_hpc_to_pfc': 'p_value'})
           .assign(direction='HPC -> PFC'),
        res[['state', 'segment_idx', 'n_samples_used', 'selected_lag', 'f_pfc_to_hpc', 'p_pfc_to_hpc']]
           .rename(columns={'f_pfc_to_hpc': 'f_stat', 'p_pfc_to_hpc': 'p_value'})
           .assign(direction='PFC -> HPC')
    ], ignore_index=True)

    long_df['fdr_sig_0p05'] = False
    for d in long_df['direction'].unique():
        m = long_df['direction'] == d
        long_df.loc[m, 'fdr_sig_0p05'] = _bh_fdr_flags_local(long_df.loc[m, 'p_value'].values, alpha=0.05)

    summary = (
        long_df.groupby(['state', 'direction'], as_index=False)
        .agg(
            n_segments_tested=('segment_idx', 'count'),
            n_nominal_p_lt_0p05=('p_value', lambda x: int((np.asarray(x) < 0.05).sum())),
            n_fdr_sig_0p05=('fdr_sig_0p05', lambda x: int(np.asarray(x, dtype=bool).sum())),
            median_selected_lag=('selected_lag', 'median'),
            median_p=('p_value', 'median'),
            median_f=('f_stat', 'median'),
            median_n_samples=('n_samples_used', 'median')
        )
    )

    return long_df, summary


if _HAS_STATSMODELS:
    sm_phasic, sm_phasic_summary = run_statsmodels_var_gc_by_state(
        phasic_theta_imfs, theta_pfc_phasic_rem_lfp, state_name='phasic', maxlags=20, ic='bic', difference=True
    )
    sm_tonic, sm_tonic_summary = run_statsmodels_var_gc_by_state(
        tonic_theta_imfs, theta_pfc_tonic_rem_lfp, state_name='tonic', maxlags=20, ic='bic', difference=True
    )

    sm_gc_results = pd.concat([sm_phasic, sm_tonic], ignore_index=True)
    sm_gc_summary = pd.concat([sm_phasic_summary, sm_tonic_summary], ignore_index=True)

    print('statsmodels VAR-GC summary:')
    display(sm_gc_summary)

    if not sm_gc_results.empty:
        vis = sm_gc_results.copy()
        vis['neglog10_p'] = -np.log10(np.clip(vis['p_value'].astype(float), 1e-300, 1.0))

        fig, ax = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)

        sns.barplot(data=sm_gc_summary, x='state', y='n_segments_tested', hue='direction', ax=ax[0])
        ax[0].set_title('Segments Tested (VAR-GC)')
        ax[0].set_ylabel('count')
        ax[0].set_xlabel('state')
        ax[0].legend(frameon=False)

        sns.barplot(data=sm_gc_summary, x='state', y='n_fdr_sig_0p05', hue='direction', ax=ax[1])
        ax[1].set_title('FDR-significant Segments (VAR-GC)')
        ax[1].set_ylabel('count')
        ax[1].set_xlabel('state')
        ax[1].legend(frameon=False)

        sns.boxplot(data=vis, x='state', y='neglog10_p', hue='direction', ax=ax[2])
        ax[2].set_title('Per-segment Strength (-log10 p)')
        ax[2].set_ylabel('-log10(p)')
        ax[2].set_xlabel('state')
        ax[2].legend(frameon=False)

        plt.show()


## Version A Extension: MNE spectral GC across all dataset

This extends Version A to process all rat folders/trials using the same folder traversal template as `extract_data_for_rat_fppsig`,
while using **raw interval-matched HPC/PFC segments** for GC input.

Rat selection:
- `rat_ids='all'` for all rats found in dataset
- `rat_ids=1` for one rat
- `rat_ids=[1,2,7]` for selected rats

In [ ]:
# Dataset-wide MNE spectral GC (raw-input), using your dataset traversal template


def _normalize_rat_selector(rat_ids):
    """Return None for all rats; else a set[str] for filtering."""
    if rat_ids is None or rat_ids == 'all':
        return None
    if isinstance(rat_ids, (int, np.integer, str)):
        return {str(rat_ids)}
    return {str(r) for r in rat_ids}


def _extract_raw_lfp_by_intervals_any(lfp_like, fs, interval):
    """Extract interval segments from ndarray or nap Tsd/TsdFrame-like containers."""
    if len(interval) == 0:
        return []

    # Convert container to 1D numpy signal
    if hasattr(lfp_like, 'values'):
        arr = np.asarray(lfp_like.values)
    elif hasattr(lfp_like, 'd'):
        arr = np.asarray(lfp_like.d)
    else:
        arr = np.asarray(lfp_like)

    arr = np.asarray(arr)
    if arr.ndim == 2:
        if arr.shape[1] == 1:
            arr = arr[:, 0]
        else:
            arr = arr.squeeze()
            if arr.ndim == 2:
                arr = arr[:, 0]
    arr = np.asarray(arr, dtype=float).squeeze()

    segs = []
    for ii in range(len(interval)):
        start_idx = int(interval.loc[ii, 'start'] * fs)
        end_idx = int(interval.loc[ii, 'end'] * fs)
        seg = arr[start_idx:end_idx]
        segs.append(np.asarray(seg, dtype=float))

    return segs


def run_mne_gc_across_dataset(
    rat_ids='all',
    base_path='/Users/amir/Desktop/for Abdel/RGS/DatabyCondition',
    preferred_conditions=("HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD", "HomeCageCG"),
    fmin=5.0,
    fmax=20.0,
    theta_band=(5.0, 12.0),
    gc_n_lags=10,
    min_epoch_sec=2.0,
    min_n_cycles=5,
    store_spectra=False,
    verbose=True,
):
    """
    Run trial-wise MNE spectral GC across dataset.

    Returns:
      trial_summary_df : one row per (rat, condition, trial, state)
      agg_summary_df   : aggregated by (state, direction)
      spectra_list     : optional per-trial spectral dicts (if store_spectra=True)
    """
    if not _HAS_MNE_GC:
        print('mne-connectivity is not available; cannot run dataset-wide MNE GC.')
        return pd.DataFrame(), pd.DataFrame(), []

    rat_filter = _normalize_rat_selector(rat_ids)

    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    if not conditions:
        print(f'No condition folders found under: {base_path}')
        return pd.DataFrame(), pd.DataFrame(), []

    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d-]+)$')

    all_rows = []
    all_specs = []

    # Optional discovery of available rats
    discovered_rats = set()
    for condition_folder in conditions:
        cpath = os.path.join(base_path, condition_folder)
        for f in os.listdir(cpath):
            full = os.path.join(cpath, f)
            if not os.path.isdir(full):
                continue
            m = folder_re.match(f)
            if m:
                discovered_rats.add(m.group(1))

    if rat_filter is None:
        rat_filter = discovered_rats

    if verbose:
        print('Selected rats:', sorted(rat_filter, key=lambda x: int(x)))

    for condition_folder in conditions:
        condition_path = os.path.join(base_path, condition_folder)

        rat_folders = []
        for f in os.listdir(condition_path):
            full = os.path.join(condition_path, f)
            if not os.path.isdir(full):
                continue
            m = folder_re.match(f)
            if not m:
                continue
            if m.group(1) in rat_filter:
                rat_folders.append((f, m))

        if not rat_folders:
            continue

        for rat_folder, m in sorted(rat_folders, key=lambda x: x[0]):
            rat_id = m.group(1)
            sd_number = m.group(2)
            condition_label = m.group(3)
            date_part = m.group(4)
            rat_path = os.path.join(condition_path, rat_folder)

            if verbose:
                print(f'Processing rat folder: {rat_folder}')

            post_trial_folders = [
                f for f in os.listdir(rat_path)
                if os.path.isdir(os.path.join(rat_path, f)) and re.search(r'Post[-_]Trial\d+', f)
            ]
            post_trial_folders = sorted(post_trial_folders)

            for post_trial_folder in post_trial_folders:
                trial_path = os.path.join(rat_path, post_trial_folder)

                lfp_hpc_file, lfp_pfc_file, state_file = None, None, None
                for file_name in os.listdir(trial_path):
                    low = file_name.lower()
                    full = os.path.join(trial_path, file_name)
                    if ('hpc_100' in low) and low.endswith('.mat'):
                        lfp_hpc_file = full
                    elif ('pfc_100' in low) and low.endswith('.mat'):
                        lfp_pfc_file = full
                    elif ('states' in low) and low.endswith('.mat'):
                        state_file = full

                if not lfp_hpc_file or not lfp_pfc_file or not state_file:
                    if verbose:
                        print(f'Missing HPC/PFC/states in {trial_path}. Skipping...')
                    continue

                trial_number_match = re.search(r'Post[-_]Trial(\d+)', post_trial_folder)
                if not trial_number_match:
                    if verbose:
                        print(f'Unable to parse trial number from {post_trial_folder}. Skipping...')
                    continue
                trial_number = int(trial_number_match.group(1))

                try:
                    lfpHPC, hypno, fs_hpc = get_data(lfp_hpc_file, state_file)
                    lfpPFC, _hypno2, fs_pfc = get_data(lfp_pfc_file, state_file, type='pfc')

                    if fs_hpc != fs_pfc:
                        if verbose:
                            print(f'FS mismatch in {post_trial_folder}: HPC={fs_hpc}, PFC={fs_pfc}. Skipping...')
                        continue

                    fs_local = int(fs_hpc)

                    try:
                        phasic_interval, tonic_interval, lfp_raw_hpc = extract_pt_intervals(lfpHPC, hypno, fs=fs_local)
                    except ValueError:
                        if verbose:
                            print(f'No REM sleep found in {post_trial_folder} (Rat {rat_id}, Cond {condition_label}).')
                        continue

                    if (len(phasic_interval) == 0) or (len(tonic_interval) == 0):
                        continue

                    # Raw interval segments
                    hpc_phasic_raw = _extract_raw_lfp_by_intervals_any(lfp_raw_hpc, fs_local, phasic_interval)
                    hpc_tonic_raw = _extract_raw_lfp_by_intervals_any(lfp_raw_hpc, fs_local, tonic_interval)
                    pfc_phasic_raw = _extract_raw_lfp_by_intervals_any(lfpPFC, fs_local, phasic_interval)
                    pfc_tonic_raw = _extract_raw_lfp_by_intervals_any(lfpPFC, fs_local, tonic_interval)

                    # Per-trial per-state MNE GC
                    out_ph = run_mne_spectral_gc_for_state(
                        hpc_phasic_raw,
                        pfc_phasic_raw,
                        state_name='phasic',
                        sfreq=fs_local,
                        fmin=fmin,
                        fmax=fmax,
                        theta_band=theta_band,
                        gc_n_lags=gc_n_lags,
                        min_epoch_sec=min_epoch_sec,
                        min_n_cycles=min_n_cycles,
                    )
                    out_to = run_mne_spectral_gc_for_state(
                        hpc_tonic_raw,
                        pfc_tonic_raw,
                        state_name='tonic',
                        sfreq=fs_local,
                        fmin=fmin,
                        fmax=fmax,
                        theta_band=theta_band,
                        gc_n_lags=gc_n_lags,
                        min_epoch_sec=min_epoch_sec,
                        min_n_cycles=min_n_cycles,
                    )

                    for out in [out_ph, out_to]:
                        row_df, spec, _epochs = out
                        if row_df is None:
                            continue

                        row_df = row_df.copy()
                        row_df['rat_id'] = int(rat_id)
                        row_df['SD'] = sd_number
                        row_df['condition'] = condition_label
                        row_df['condition_folder'] = condition_folder
                        row_df['date'] = date_part
                        row_df['trial'] = trial_number
                        row_df['post_trial_folder'] = post_trial_folder
                        all_rows.append(row_df)

                        if store_spectra:
                            spec_meta = {
                                'rat_id': int(rat_id),
                                'SD': sd_number,
                                'condition': condition_label,
                                'condition_folder': condition_folder,
                                'date': date_part,
                                'trial': trial_number,
                                'post_trial_folder': post_trial_folder,
                                'state': spec['state'],
                                'freqs': spec['freqs'],
                                'gc_hpc_to_pfc': spec['gc_hpc_to_pfc'],
                                'gc_pfc_to_hpc': spec['gc_pfc_to_hpc'],
                                'valid_spec_mask': spec['valid_spec_mask'],
                                'meta': spec['meta'],
                            }
                            all_specs.append(spec_meta)

                except FileNotFoundError:
                    if verbose:
                        print(f'Data not found in {trial_path}. Skipping...')
                    continue
                except Exception as e:
                    if verbose:
                        print(f'Error in {trial_path}: {e}')
                    continue

    if len(all_rows) == 0:
        print('No MNE GC data extracted.')
        return pd.DataFrame(), pd.DataFrame(), all_specs

    trial_summary_df = pd.concat(all_rows, ignore_index=True)

    long_df = trial_summary_df.melt(
        id_vars=[c for c in trial_summary_df.columns if c not in ['theta_gc_hpc_to_pfc', 'theta_gc_pfc_to_hpc']],
        value_vars=['theta_gc_hpc_to_pfc', 'theta_gc_pfc_to_hpc'],
        var_name='direction',
        value_name='theta_gc'
    )
    long_df['direction'] = long_df['direction'].replace({
        'theta_gc_hpc_to_pfc': 'HPC -> PFC',
        'theta_gc_pfc_to_hpc': 'PFC -> HPC',
    })

    agg_summary_df = (
        long_df.groupby(['state', 'direction'], as_index=False)
        .agg(
            n_trials=('theta_gc', 'count'),
            mean_theta_gc=('theta_gc', 'mean'),
            median_theta_gc=('theta_gc', 'median'),
            std_theta_gc=('theta_gc', 'std'),
        )
    )

    return trial_summary_df, agg_summary_df, all_specs


# --------- Usage examples ---------
# 1) All rats:
# mne_trial_summary_all, mne_agg_all, _ = run_mne_gc_across_dataset(rat_ids='all', store_spectra=False)
# display(mne_agg_all)

# 2) One rat:
# mne_trial_summary_r1, mne_agg_r1, _ = run_mne_gc_across_dataset(rat_ids=1, store_spectra=False)
# display(mne_agg_r1)

# 3) Selected rats:
# mne_trial_summary_sel, mne_agg_sel, _ = run_mne_gc_across_dataset(rat_ids=[1,2,7], store_spectra=False)
# display(mne_agg_sel)

In [ ]:
mne_trial_summary_all, mne_agg_all, _ = run_mne_gc_across_dataset(rat_ids='all', store_spectra=False)
display(mne_agg_all)

In [ ]:
# Optional plotting helper for dataset-wide MNE results

def plot_mne_dataset_gc(mne_trial_summary_df):
    if mne_trial_summary_df is None or len(mne_trial_summary_df) == 0:
        print('No dataset-wide MNE summary to plot.')
        return

    long_df = mne_trial_summary_df.melt(
        id_vars=[c for c in mne_trial_summary_df.columns if c not in ['theta_gc_hpc_to_pfc', 'theta_gc_pfc_to_hpc']],
        value_vars=['theta_gc_hpc_to_pfc', 'theta_gc_pfc_to_hpc'],
        var_name='direction',
        value_name='theta_gc'
    )
    long_df['direction'] = long_df['direction'].replace({
        'theta_gc_hpc_to_pfc': 'HPC -> PFC',
        'theta_gc_pfc_to_hpc': 'PFC -> HPC',
    })

    fig, ax = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)

    # Mean +/- spread by state & direction
    sns.barplot(data=long_df, x='state', y='theta_gc', hue='direction', ax=ax[0], errorbar='sd')
    ax[0].set_title('Dataset-wide Theta GC (mean ± sd)')
    ax[0].set_xlabel('state')
    ax[0].set_ylabel('theta GC')
    ax[0].legend(frameon=False)

    # Trial-level distributions
    sns.boxplot(data=long_df, x='state', y='theta_gc', hue='direction', ax=ax[1])
    ax[1].set_title('Trial-level Theta GC distribution')
    ax[1].set_xlabel('state')
    ax[1].set_ylabel('theta GC')
    ax[1].legend(frameon=False)

    # Directionality index by state
    dir_df = mne_trial_summary_df.copy()
    sns.boxplot(data=dir_df, x='state', y='theta_directionality_h2p_minus_p2h', ax=ax[2])
    ax[2].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[2].set_title('Directionality Index (HPC->PFC minus PFC->HPC)')
    ax[2].set_xlabel('state')
    ax[2].set_ylabel('directionality')

    plt.show()


# Example after running run_mne_gc_across_dataset(...):
plot_mne_dataset_gc(mne_trial_summary_all)


### positive

In [ ]:
mne_trial_summary_sel, mne_agg_sel, _ = run_mne_gc_across_dataset(rat_ids=[2, 4, 7, 8], store_spectra=False)
display(mne_agg_sel)

In [ ]:
def plot_trial_level_gc(mne_trial_summary_df, save_path=None):
    """
    Publication-quality trial-level Theta GC plot with statistical annotations.
    Each dot = one trial, colored by rat on left panel only.

    Stats: Mann-Whitney U test (phasic vs tonic) for each direction,
           Wilcoxon signed-rank (HPC->PFC vs PFC->HPC) within each state,
           and directionality tests.
    """
    from scipy.stats import mannwhitneyu, wilcoxon
    from matplotlib.lines import Line2D

    if mne_trial_summary_df is None or len(mne_trial_summary_df) == 0:
        print('No data to plot.')
        return

    df = mne_trial_summary_df.copy()

    # Melt to long format
    long_df = df.melt(
        id_vars=[c for c in df.columns if c not in ['theta_gc_hpc_to_pfc', 'theta_gc_pfc_to_hpc']],
        value_vars=['theta_gc_hpc_to_pfc', 'theta_gc_pfc_to_hpc'],
        var_name='direction', value_name='theta_gc'
    )
    long_df['direction'] = long_df['direction'].replace({
        'theta_gc_hpc_to_pfc': r'HPC $\rightarrow$ PFC',
        'theta_gc_pfc_to_hpc': r'PFC $\rightarrow$ HPC',
    })

    # Directionality
    df['directionality'] = df['theta_directionality_h2p_minus_p2h']

    rats = sorted(df['rat_id'].unique())
    _no_green = ['#4C72B0', '#DD8452', '#8172B3', '#C44E52',
                 '#DA8BC3', '#937860', '#CCB974', '#64B5CD']
    rat_palette = {r: _no_green[i % len(_no_green)] for i, r in enumerate(rats)}

    def _fmt_p(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        else:
            return 'n.s.'

    def _annotate_bracket(ax, x1, x2, y, p_text, lw=1.0, h=0.01):
        dy = h * (ax.get_ylim()[1] - ax.get_ylim()[0])
        ax.plot([x1, x1, x2, x2], [y, y + dy, y + dy, y], color='black', lw=lw, clip_on=False)
        ax.text((x1 + x2) / 2, y + dy, p_text, ha='center', va='bottom', fontsize=9)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    # ---- Panel 1: GC by state & direction ----
    ax = axes[0]
    sns.boxplot(data=long_df, x='state', y='theta_gc', hue='direction',
                ax=ax, width=0.5, showfliers=False,
                boxprops=dict(alpha=0.4), whiskerprops=dict(alpha=0.4),
                medianprops=dict(color='black', linewidth=1.5),
                capprops=dict(alpha=0.4))
    sns.stripplot(data=long_df, x='state', y='theta_gc', hue='direction',
                  dodge=True, ax=ax, size=5, alpha=0.7, jitter=0.12,
                  palette=['#4C72B0', '#DD8452'], legend=False)
    ax.set_title('Trial-level Theta GC', fontsize=13, fontweight='bold')
    ax.set_ylabel('Theta GC', fontsize=11)
    ax.set_xlabel('')
    ax.legend(frameon=False, fontsize=9, loc='center right')
    sns.despine(ax=ax)

    # Stats: phasic vs tonic within each direction
    ymax = long_df['theta_gc'].max()
    offset = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05
    for dir_idx, direction in enumerate([r'HPC $\rightarrow$ PFC', r'PFC $\rightarrow$ HPC']):
        vals_ph = long_df[(long_df['state'] == 'phasic') & (long_df['direction'] == direction)]['theta_gc']
        vals_to = long_df[(long_df['state'] == 'tonic') & (long_df['direction'] == direction)]['theta_gc']
        if len(vals_ph) > 0 and len(vals_to) > 0:
            _, p = mannwhitneyu(vals_ph, vals_to, alternative='two-sided')
            x1 = -0.25 + dir_idx * 0.25 + 0.02
            x2 = 0.75 + dir_idx * 0.25 + 0.02
            y_bar = ymax + offset + dir_idx * offset * 2.5
            _annotate_bracket(ax, x1, x2, y_bar, f'{_fmt_p(p)} p={p:.3f}', h=0.008)

    # Stats: HPC->PFC vs PFC->HPC within each state
    for state_idx, state in enumerate(['phasic', 'tonic']):
        vals_h2p = long_df[(long_df['state'] == state) & (long_df['direction'] == r'HPC $\rightarrow$ PFC')]['theta_gc']
        vals_p2h = long_df[(long_df['state'] == state) & (long_df['direction'] == r'PFC $\rightarrow$ HPC')]['theta_gc']
        if len(vals_h2p) > 0 and len(vals_p2h) > 0:
            n_common = min(len(vals_h2p), len(vals_p2h))
            if n_common >= 5:
                _, p = wilcoxon(vals_h2p.values[:n_common], vals_p2h.values[:n_common])
            else:
                _, p = mannwhitneyu(vals_h2p, vals_p2h, alternative='two-sided')
            x1 = state_idx - 0.13
            x2 = state_idx + 0.13
            y_bar = ymax + offset * 6
            _annotate_bracket(ax, x1, x2, y_bar, f'{_fmt_p(p)}', h=0.008)

    # ---- Panel 2: Directionality index ----
    ax = axes[1]
    sns.boxplot(data=df, x='state', y='directionality',
                ax=ax, width=0.35, showfliers=False,
                boxprops=dict(alpha=0.4, facecolor='#8da0cb'),
                whiskerprops=dict(alpha=0.4),
                medianprops=dict(color='black', linewidth=1.5),
                capprops=dict(alpha=0.4))
    sns.stripplot(data=df, x='state', y='directionality',
                  ax=ax, size=5, alpha=0.75,
                  jitter=0.1, dodge=False, color='black')
    ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title(r'Directionality (HPC$\rightarrow$PFC minus PFC$\rightarrow$HPC)',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Directionality index', fontsize=11)
    ax.set_xlabel('')
    sns.despine(ax=ax)

    # Stats: phasic vs tonic directionality
    dir_ph = df[df['state'] == 'phasic']['directionality']
    dir_to = df[df['state'] == 'tonic']['directionality']
    if len(dir_ph) > 0 and len(dir_to) > 0:
        _, p = mannwhitneyu(dir_ph, dir_to, alternative='two-sided')
        ymax_d = max(dir_ph.max(), dir_to.max())
        offset_d = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05
        _annotate_bracket(ax, 0, 1, ymax_d + offset_d, f'{_fmt_p(p)} p={p:.3f}', h=0.008)

    # One-sample Wilcoxon against 0 for each state
    for state_idx, state in enumerate(['phasic', 'tonic']):
        vals = df[df['state'] == state]['directionality'].dropna()
        if len(vals) >= 5:
            _, p_one = wilcoxon(vals)
            offset_d = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05
            ax.text(state_idx, ax.get_ylim()[0] + offset_d * 0.5,
                    f'vs 0: {_fmt_p(p_one)}',
                    ha='center', va='bottom', fontsize=8, fontstyle='italic', color='gray')

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()

In [ ]:
# After running: mne_trial_summary_sel, mne_agg_sel, _ = run_mne_gc_across_dataset(...)

plot_trial_level_gc(mne_trial_summary_sel, save_path='figures/trial_level_gc_positive.png')

In [ ]:
plot_mne_dataset_gc(mne_trial_summary_sel)

### control

In [ ]:
mne_trial_summary_sel, mne_agg_sel, _ = run_mne_gc_across_dataset(rat_ids=[1, 3, 6, 9], store_spectra=False)
display(mne_agg_sel)

In [ ]:
def plot_trial_level_gc(mne_trial_summary_df, save_path=None):
    """
    Publication-quality trial-level Theta GC plot with statistical annotations.
    Each dot = one trial, colored by rat on left panel only.

    Stats: Mann-Whitney U test (phasic vs tonic) for each direction,
           Wilcoxon signed-rank (HPC->PFC vs PFC->HPC) within each state,
           and directionality tests.
    """
    from scipy.stats import mannwhitneyu, wilcoxon
    from matplotlib.lines import Line2D

    if mne_trial_summary_df is None or len(mne_trial_summary_df) == 0:
        print('No data to plot.')
        return

    df = mne_trial_summary_df.copy()

    # Melt to long format
    long_df = df.melt(
        id_vars=[c for c in df.columns if c not in ['theta_gc_hpc_to_pfc', 'theta_gc_pfc_to_hpc']],
        value_vars=['theta_gc_hpc_to_pfc', 'theta_gc_pfc_to_hpc'],
        var_name='direction', value_name='theta_gc'
    )
    long_df['direction'] = long_df['direction'].replace({
        'theta_gc_hpc_to_pfc': r'HPC $\rightarrow$ PFC',
        'theta_gc_pfc_to_hpc': r'PFC $\rightarrow$ HPC',
    })

    # Directionality
    df['directionality'] = df['theta_directionality_h2p_minus_p2h']

    rats = sorted(df['rat_id'].unique())
    _no_green = ['#4C72B0', '#DD8452', '#8172B3', '#C44E52',
                 '#DA8BC3', '#937860', '#CCB974', '#64B5CD']
    rat_palette = {r: _no_green[i % len(_no_green)] for i, r in enumerate(rats)}

    def _fmt_p(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        else:
            return 'n.s.'

    def _annotate_bracket(ax, x1, x2, y, p_text, lw=1.0, h=0.01):
        dy = h * (ax.get_ylim()[1] - ax.get_ylim()[0])
        ax.plot([x1, x1, x2, x2], [y, y + dy, y + dy, y], color='black', lw=lw, clip_on=False)
        ax.text((x1 + x2) / 2, y + dy, p_text, ha='center', va='bottom', fontsize=9)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    # ---- Panel 1: GC by state & direction ----
    ax = axes[0]
    sns.boxplot(data=long_df, x='state', y='theta_gc', hue='direction',
                ax=ax, width=0.5, showfliers=False,
                boxprops=dict(alpha=0.4), whiskerprops=dict(alpha=0.4),
                medianprops=dict(color='black', linewidth=1.5),
                capprops=dict(alpha=0.4))
    sns.stripplot(data=long_df, x='state', y='theta_gc', hue='direction',
                  dodge=True, ax=ax, size=5, alpha=0.7, jitter=0.12,
                  palette=['#4C72B0', '#DD8452'], legend=False)
    ax.set_title('Trial-level Theta GC', fontsize=13, fontweight='bold')
    ax.set_ylabel('Theta GC', fontsize=11)
    ax.set_xlabel('')
    ax.legend(frameon=False, fontsize=9, loc='center right')
    sns.despine(ax=ax)

    # Stats: phasic vs tonic within each direction
    ymax = long_df['theta_gc'].max()
    offset = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05
    for dir_idx, direction in enumerate([r'HPC $\rightarrow$ PFC', r'PFC $\rightarrow$ HPC']):
        vals_ph = long_df[(long_df['state'] == 'phasic') & (long_df['direction'] == direction)]['theta_gc']
        vals_to = long_df[(long_df['state'] == 'tonic') & (long_df['direction'] == direction)]['theta_gc']
        if len(vals_ph) > 0 and len(vals_to) > 0:
            _, p = mannwhitneyu(vals_ph, vals_to, alternative='two-sided')
            x1 = -0.25 + dir_idx * 0.25 + 0.02
            x2 = 0.75 + dir_idx * 0.25 + 0.02
            y_bar = ymax + offset + dir_idx * offset * 2.5
            _annotate_bracket(ax, x1, x2, y_bar, f'{_fmt_p(p)} p={p:.3f}', h=0.008)

    # Stats: HPC->PFC vs PFC->HPC within each state
    for state_idx, state in enumerate(['phasic', 'tonic']):
        vals_h2p = long_df[(long_df['state'] == state) & (long_df['direction'] == r'HPC $\rightarrow$ PFC')]['theta_gc']
        vals_p2h = long_df[(long_df['state'] == state) & (long_df['direction'] == r'PFC $\rightarrow$ HPC')]['theta_gc']
        if len(vals_h2p) > 0 and len(vals_p2h) > 0:
            n_common = min(len(vals_h2p), len(vals_p2h))
            if n_common >= 5:
                _, p = wilcoxon(vals_h2p.values[:n_common], vals_p2h.values[:n_common])
            else:
                _, p = mannwhitneyu(vals_h2p, vals_p2h, alternative='two-sided')
            x1 = state_idx - 0.13
            x2 = state_idx + 0.13
            y_bar = ymax + offset * 6
            _annotate_bracket(ax, x1, x2, y_bar, f'{_fmt_p(p)}', h=0.008)

    # ---- Panel 2: Directionality index ----
    ax = axes[1]
    sns.boxplot(data=df, x='state', y='directionality',
                ax=ax, width=0.35, showfliers=False,
                boxprops=dict(alpha=0.4, facecolor='#8da0cb'),
                whiskerprops=dict(alpha=0.4),
                medianprops=dict(color='black', linewidth=1.5),
                capprops=dict(alpha=0.4))
    sns.stripplot(data=df, x='state', y='directionality',
                  ax=ax, size=5, alpha=0.75,
                  jitter=0.1, dodge=False, color='black')
    ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title(r'Directionality (HPC$\rightarrow$PFC minus PFC$\rightarrow$HPC)',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Directionality index', fontsize=11)
    ax.set_xlabel('')
    sns.despine(ax=ax)

    # Stats: phasic vs tonic directionality
    dir_ph = df[df['state'] == 'phasic']['directionality']
    dir_to = df[df['state'] == 'tonic']['directionality']
    if len(dir_ph) > 0 and len(dir_to) > 0:
        _, p = mannwhitneyu(dir_ph, dir_to, alternative='two-sided')
        ymax_d = max(dir_ph.max(), dir_to.max())
        offset_d = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05
        _annotate_bracket(ax, 0, 1, ymax_d + offset_d, f'{_fmt_p(p)} p={p:.3f}', h=0.008)

    # One-sample Wilcoxon against 0 for each state
    for state_idx, state in enumerate(['phasic', 'tonic']):
        vals = df[df['state'] == state]['directionality'].dropna()
        if len(vals) >= 5:
            _, p_one = wilcoxon(vals)
            offset_d = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05
            ax.text(state_idx, ax.get_ylim()[0] + offset_d * 0.5,
                    f'vs 0: {_fmt_p(p_one)}',
                    ha='center', va='bottom', fontsize=8, fontstyle='italic', color='gray')

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()

In [ ]:
# After running: mne_trial_summary_sel, mne_agg_sel, _ = run_mne_gc_across_dataset(...)

plot_trial_level_gc(mne_trial_summary_sel, save_path='figures/trial_level_gc_control.png')

In [ ]:
plot_mne_dataset_gc(mne_trial_summary_sel)